# Weather-station imputation

This notebook processes one prepared DWD hourly-temperature file from `WEATHER_DATA_DIR`. The current DWD source files are semicolon-delimited `.txt` files, which the domain loader reads directly. It uses the station's genuine missing values, **IQR outlier detection**, and **min–max standardization**, deliberately differing from the eye-tracking example. The source file remains untouched.

In [1]:
from pathlib import Path
import os
import sys
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'pyproject.toml').is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError('Start Jupyter from inside the repository.')
    PROJECT_ROOT = PROJECT_ROOT.parent
for line in (PROJECT_ROOT / '.env').read_text(encoding='utf-8').splitlines():
    line = line.strip()
    if line and not line.startswith('#') and '=' in line:
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip(chr(34)).strip(chr(39)))
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from gap_imputation_benchmark.algorithm import DomainImputationConfig, RunMetadata, impute_with_rf_selector
from gap_imputation_benchmark.domains.weather.loaders import load_dwd_station

DATA_ROOT = Path(os.environ['WEATHER_DATA_DIR'])
OUTPUT_DIR = PROJECT_ROOT / 'notebooks' / 'algorithm' / 'weather' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXECUTOR_NAME = 'Oliver'
EXECUTOR_INFO_PATH = PROJECT_ROOT / 'data' / 'person_info_examples' / 'executor_example.json'
RESPONSIBLE_PERSON_NAME = 'Jenny'
RESPONSIBLE_PERSON_INFO_PATH = PROJECT_ROOT / 'data' / 'person_info_examples' / 'executor_responsible_person_example.json'
EXECUTION_NOTEBOOK_PATH = PROJECT_ROOT / 'notebooks' / 'algorithm' / 'weather' / '01_weather_csv_imputation.ipynb'


In [2]:
prepared_files = sorted((DATA_ROOT / 'processed' / 'temperature_2000_2025').glob('*.txt'))
if not prepared_files:
    raise FileNotFoundError('No prepared DWD station file found. Run the Weather benchmark preparation first or provide processed/temperature_2000_2025/.')
SOURCE_FILE = prepared_files[0]
frame = load_dwd_station(
    SOURCE_FILE, start='2000-01-01 00:00', end='2025-12-31 23:00', source_file=str(SOURCE_FILE),
)[['timestamp', 'gaze_x', 'is_valid']].copy()
frame = frame.rename(columns={'gaze_x': 'temperature_c'})
VALUE_COLUMN = 'temperature_c'
frame.head(), SOURCE_FILE.name

(            timestamp  temperature_c  is_valid
 0 2000-01-01 00:00:00            0.4      True
 1 2000-01-01 01:00:00            0.4      True
 2 2000-01-01 02:00:00            0.6      True
 3 2000-01-01 03:00:00            1.0      True
 4 2000-01-01 04:00:00            1.1      True,
 'produkt_tu_stunde_19470101_20251231_04271.txt')

In [3]:
n_missing = int(frame[VALUE_COLUMN].isna().sum())
if n_missing == 0:
    raise ValueError('The selected station has no missing values; choose another station file.')
print(f'Using {n_missing} genuine missing hourly values from {SOURCE_FILE.name}.')

Using 296 genuine missing hourly values from produkt_tu_stunde_19470101_20251231_04271.txt.


In [4]:
config = DomainImputationConfig(
    domain='weather', timestamp_col='timestamp', validity_col='is_valid',
    outlier_method='iqr', outlier_threshold=2.5, standardization_method='minmax',
)
imputed_frame, provenance = impute_with_rf_selector(
    frame, VALUE_COLUMN, config=config,
    run_metadata=RunMetadata(
        executor_name=EXECUTOR_NAME, executor_info_path=EXECUTOR_INFO_PATH,
        executor_responsible_person=RESPONSIBLE_PERSON_NAME,
        executor_responsible_person_info_path=RESPONSIBLE_PERSON_INFO_PATH,
        execution_notebook_path=EXECUTION_NOTEBOOK_PATH,
        comment=f'Weather example from {SOURCE_FILE.name}.',
    ),
    input_path=SOURCE_FILE, output_path=OUTPUT_DIR / 'weather_imputed.csv',
    provenance_path=OUTPUT_DIR / 'weather_provenance.json',
)
provenance['summary']

{'gaps_before_outlier_detection': 19,
 'gaps_after_outlier_detection': 19,
 'filled_gap_count': 19,
 'skipped_gap_count': 0}

## Review

Weather requires calendar timestamps because the candidate portfolio includes calendar-year seasonal references. The run report documents whether that candidate was applicable, the selected method, and all fallbacks. Outputs are local derived files and are ignored by Git.